# 🫁 RadiAssist: Chest X-ray Triage, training on a Colab T4

This notebook runs the **whole ML side** of the project on a free Colab T4 GPU:

| Step | What happens | Time (T4) |
|---|---|---|
| 1 | Mount Google Drive, get the code, install | 3 min |
| 2 | Kaggle credentials | 1 min |
| 3 | Download NIH ChestX-ray14 (42 GB) and prepare a 7 GB cache, saved to Drive | ~40 min first time, ~3 min after |
| 4 | Look at the data (EDA) | 1 min |
| 5 | MLflow experiment tracking | 1 min |
| 6 | Sanity run (catches bugs before the long run) | 5 min |
| 7 | **Full training**: DenseNet-121, mixed precision | ~1.5-2.5 h |
| 8 | Evaluation: AUROC with 95% CI, calibration, Grad-CAM, subgroups | 10 min |
| 9 | Export to ONNX, register in MLflow, quality-gate promotion | 3 min |
| 10 | Test the API on a real X-ray, inside Colab | 2 min |
| 11 | Publish the model and deploy the web app to Hugging Face | 5 min |

**Before you start:** `Runtime → Change runtime type → T4 GPU`.
Everything important (dataset cache, checkpoints, MLflow, reports) is written to **Google Drive**, so a disconnect never loses work.

## Step 0: check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1: Drive, code, dependencies
Set `REPO_URL` to your GitHub repository. Alternatively, zip the project folder, upload it to `MyDrive/cxr-triage/chest-xray-triage.zip` and leave `REPO_URL` unchanged.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cxr-triage"      # all persistent outputs live here
os.makedirs(DRIVE, exist_ok=True)

REPO_URL = "https://github.com/dhiman-13/chest-xray-triage.git"
PROJECT = "/content/chest-xray-triage"
ZIP_ON_DRIVE = f"{DRIVE}/chest-xray-triage.zip"

if not os.path.exists(PROJECT):
    if "YOUR-USERNAME" not in REPO_URL:
        !git clone -q {REPO_URL} {PROJECT}
    elif os.path.exists(ZIP_ON_DRIVE):
        !unzip -q -o {ZIP_ON_DRIVE} -d /content
    else:
        raise SystemExit("Set REPO_URL, or upload chest-xray-triage.zip to " + DRIVE)
%cd {PROJECT}

In [ ]:
# torch / torchvision are already installed on Colab and are not re-downloaded
!pip install -q -r requirements.txt
!pip install -q -e .
# Make the project importable both here and in every `!python -m cxr...` subprocess
import sys
os.environ["PYTHONPATH"] = f"{PROJECT}/src:{PROJECT}"
for p in (f"{PROJECT}/src", PROJECT):
    if p not in sys.path:
        sys.path.insert(0, p)
!python -c "import cxr, timm, lightning, hydra, mlflow, cv2, onnx; print('All packages OK')"
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


## Step 2: Kaggle credentials
1. kaggle.com → profile picture → **Settings** → **API** → create a token.
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar and add, with notebook access switched on:
   * **New token type** (starts with `KGAT_`): one secret, `KAGGLE_API_TOKEN` = the token
   * **Old type** (`kaggle.json` file with username + key): `KAGGLE_USERNAME` and `KAGGLE_KEY`
3. Open https://www.kaggle.com/datasets/nih-chest-xrays/data once while logged in, so your account can download it.


In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("KAGGLE_API_TOKEN")
if token:                                   # new-style token (KGAT_...)
    !pip install -q -U "kaggle>=1.8"
    os.environ["KAGGLE_API_TOKEN"] = token
    print("Using Kaggle API token")
else:                                       # classic kaggle.json username + key
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY")
    print("Kaggle user:", os.environ["KAGGLE_USERNAME"])


## Step 3: dataset (NIH ChestX-ray14)
112,120 frontal X-rays of 30,805 patients with 14 labels. The prepare script reads the PNGs **straight from the zip** (no extraction, so it fits on Colab's disk), resizes them to 256x256 and packs them into a single `images.npy` file.
It uses the **official patient-wise test split**: no patient appears in both train and test.

`SUBSET = 0.1` gives a 10% version in ~10 minutes, useful to try the pipeline. Use `1.0` for the real result.

In [ ]:
SUBSET = 1.0
tag = "full" if SUBSET >= 1 else f"subset{int(SUBSET*100)}"
DATA_DRIVE = f"{DRIVE}/data/processed_{tag}"
DATA = f"/content/data/processed_{tag}"   # local copy: much faster to read than Drive
ZIP = "/content/raw/data.zip"

if os.path.exists(f"{DATA}/images.npy"):
    print("Already on local disk")
elif os.path.exists(f"{DATA_DRIVE}/images.npy"):
    print("Copying prepared dataset from Drive...")
    !mkdir -p /content/data && cp -r "{DATA_DRIVE}" /content/data/
else:
    if not os.path.exists(ZIP):
        !kaggle datasets download -d nih-chest-xrays/data -p /content/raw
    !python -m cxr.data.prepare_nih --source {ZIP} --out {DATA} --subset {SUBSET} --workers 2
    if not os.path.exists(f"{DATA}/images.npy"):
        raise SystemExit("Preparation failed (see the error above). The zip is kept, so just fix and re-run this cell.")
    !rm -f {ZIP}
    !mkdir -p "{DRIVE}/data" && cp -r {DATA} "{DRIVE}/data/"
!cat {DATA}/summary.json | head -30


## Step 4: look at the data

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from cxr.labels import LABELS
meta = pd.read_csv(f"{DATA}/metadata.csv")
images = np.load(f"{DATA}/images.npy", mmap_mode="r")
print(meta["split"].value_counts().to_string(), "\n")
print("patients per split:", meta.groupby("split")["patient_id"].nunique().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
prev = meta[meta.split == "train"][LABELS].mean().sort_values()
prev.plot.barh(ax=ax[0], color="#0d9488"); ax[0].set_title("Label prevalence (train): heavy class imbalance")
meta[LABELS].sum(1).value_counts().sort_index().plot.bar(ax=ax[1], color="#6366f1")
ax[1].set_title("Findings per image (0 = 'No Finding')"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, (_, r) in zip(axes.ravel(), meta.sample(10, random_state=1).iterrows()):
    ax.imshow(images[r.idx], cmap="gray"); ax.axis("off")
    ax.set_title(r.findings.replace("|", ", ")[:30] + f"\n{r.sex}, {r.age:.0f}y, {r.view}", fontsize=8)
plt.tight_layout(); plt.show()

## Step 5: experiment tracking (MLflow)
* **Option A (recommended, shareable):** create a free repository on https://dagshub.com, which gives you a hosted MLflow server. Interviewers can then open your experiment dashboard from a link. Add Colab secrets `DAGSHUB_USER` and `DAGSHUB_TOKEN`, and set `DAGSHUB_REPO`.
* **Option B:** a local MLflow database on your Drive.

In [ ]:
USE_DAGSHUB = False
DAGSHUB_REPO = "dhiman-13/chest-xray-triage"

if USE_DAGSHUB:
    os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_REPO}.mlflow"
    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    ARTIFACTS = "null"
else:
    os.makedirs(f"{DRIVE}/mlflow", exist_ok=True)
    os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE}/mlflow/mlflow.db"
    ARTIFACTS = f"{DRIVE}/mlflow/artifacts"

OUT = f"{DRIVE}/outputs"        # checkpoints + last_run.json
REPORTS = f"{DRIVE}/reports"    # evaluation results
BUNDLE = f"{DRIVE}/bundle"      # deployable model
print(os.environ["MLFLOW_TRACKING_URI"])

## Step 6: sanity run (~5 min)
Trains on 3% of the data for one epoch. If this crashes, the long run would too.

In [ ]:
!python -m cxr.train data.dir={DATA} paths.output_dir=/content/sanity mlflow.experiment=cxr-sanity \
    mlflow.artifact_location={ARTIFACTS} train.epochs=1 train.limit_train_batches=0.03 train.limit_val_batches=0.1

## Step 7: full training
DenseNet-121 pretrained on ImageNet (the CheXNet recipe), AdamW + warm-up + cosine LR, 16-bit mixed precision, class-weighted BCE, early stopping on validation macro-AUROC.
Checkpoints go to Drive every epoch.

> Keep the browser tab open. If Colab disconnects, run **Step 1, Step 3, Step 5** again and then the *resume* cell below.

In [ ]:
!python -m cxr.train data.dir={DATA} paths.output_dir={OUT} mlflow.artifact_location={ARTIFACTS} \
    model=densenet121 train.epochs=12 train.batch_size=64

In [ ]:
# ONLY if training was interrupted: continue from the last saved epoch
# !python -m cxr.train data.dir={DATA} paths.output_dir={OUT} mlflow.artifact_location={ARTIFACTS} \
#     model=densenet121 train.epochs=12 train.batch_size=64 train.resume_from={OUT}/checkpoints/densenet121/last.ckpt

## Step 8: evaluation
* Per-class **AUROC with 95% bootstrap confidence intervals**, AUPRC, sensitivity at 90% specificity
* **Temperature scaling** (calibration) and **per-class thresholds for 90% sensitivity**, both fit on *validation* data only
* **Subgroup analysis** (sex, age, view), **Grad-CAM** heatmaps, comparison with published baselines

In [ ]:
!python -m cxr.evaluate --run-info {OUT}/last_run.json --out {REPORTS}
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{REPORTS}/RESULTS.md").read().split("![")[0]))
for f in ["auroc_vs_baselines", "roc_curves", "reliability", "gradcam"]:
    display(Image(f"{REPORTS}/figures/{f}.png"))

### (Optional) compare architectures
Each run is logged to MLflow, so you can compare them side by side. Budget ~2 h each on a T4.

In [ ]:
# !python -m cxr.train data.dir={DATA} paths.output_dir={OUT}/effb3 mlflow.artifact_location={ARTIFACTS} model=efficientnet_b3 train.batch_size=32
# !python -m cxr.evaluate --run-info {OUT}/effb3/last_run.json --out {REPORTS}/effb3

## Step 9: export, register, promote
* **Export:** ONNX graph with calibrated probabilities and class-activation heatmaps built in; parity-checked against PyTorch
* **Register:** a new version in the MLflow Model Registry (alias `challenger`)
* **Promote:** it becomes `champion` only if its test AUROC beats the current champion (quality gate)

In [ ]:
!python -m cxr.export --run-info {OUT}/last_run.json --reports {REPORTS} --out {BUNDLE}
!python -m cxr.registry register --run-info {OUT}/last_run.json --bundle {BUNDLE}
!python -m cxr.registry promote --run-info {OUT}/last_run.json
!ls -lh {BUNDLE}

## Step 10: test the real API on a test X-ray (inside Colab)

In [ ]:
!pip install -q fastapi python-multipart prometheus-client httpx "uvicorn[standard]"
os.environ["MODEL_DIR"] = BUNDLE
os.environ["DB_PATH"] = "/content/predictions.db"
import cv2, sys
sys.path.insert(0, PROJECT)
from fastapi.testclient import TestClient
from serving.app.main import app

row = meta[(meta.split == "test") & (meta.Effusion == 1)].iloc[0]
ok, png = cv2.imencode(".png", np.asarray(images[row.idx]))
with TestClient(app) as client:
    r = client.post("/api/predict", files={"file": ("xray.png", png.tobytes(), "image/png")}).json()
print("Ground truth:", row.findings)
print("Triage:", r["triage"].upper(), "| flagged:", r["flagged"], "| latency:", r["latency_ms"], "ms")
for f in r["findings"][:5]:
    print(f"  {f['label']:20s} p={f['probability']:.3f}  threshold={f['threshold']:.3f}")

## Step 11: publish and deploy (Hugging Face, free)
1. Create a free account at https://huggingface.co → Settings → Access Tokens → **New token (write)**
2. Add Colab secret `HF_TOKEN`, and set `HF_USER` below.

This uploads the model bundle to a **model repo** and deploys the web app (API + frontend) as a **Docker Space**. The Space downloads the model on boot.
Your public link will be `https://huggingface.co/spaces/<HF_USER>/radiassist`. Put it on your CV.

In [ ]:
HF_USER = "YOUR-HF-USERNAME"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL_REPO, SPACE = f"{HF_USER}/cxr-triage-model", f"{HF_USER}/radiassist"
!python -m cxr.registry publish --bundle {BUNDLE} --repo {MODEL_REPO}
!python deploy/deploy_space.py --space {SPACE} --model-repo {MODEL_REPO}
print(f"\nApp (first build takes ~5 min): https://huggingface.co/spaces/{SPACE}")

## Done 🎉
* **Live app:** `https://huggingface.co/spaces/<HF_USER>/radiassist`
* **Experiments:** your DagsHub MLflow UI (or `mlflow ui --backend-store-uri sqlite:///...` locally)
* **Results for your CV/README:** `MyDrive/cxr-triage/reports/RESULTS.md` and `figures/`
* **Monitoring:** the app's *Monitoring* tab, `/metrics` for Prometheus, and the daily drift GitHub Action